### Após a EDA, vamos começar a trabalhar no pré-processamento dos dados raw

#### A ideia é que não precisemos fazer as relações e cruzamentos toda hora, visando facilitar as análises posteriores. Além disso, alguns ajustes são necessários pela própria natureza do que analisamos na EDA preliminar (01_eda_preliminar)

In [21]:
import pandas as pd

In [22]:
orderItems = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
orderDataset = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
nameTranslation = pd.read_csv("../data/raw/product_category_name_translation.csv")
orderReviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orderPayments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

Aqui, vamos filtrar os status de order_status, converter as strings em datas e calcular o nosso alvo de dias_atrasados

In [23]:
entregues = orderDataset[orderDataset["order_status"] == "delivered"].copy()
entregues = entregues[entregues["order_delivered_customer_date"].notna()]

entregues["order_delivered_customer_date"] = pd.to_datetime(entregues["order_delivered_customer_date"])
entregues["order_estimated_delivery_date"] = pd.to_datetime(entregues["order_estimated_delivery_date"])

entregues["dias_atrasados"] = (entregues["order_delivered_customer_date"] - entregues["order_estimated_delivery_date"]).dt.total_seconds() / 86400

Agora, tratar aqueles 610 produtos sem categoria que encontramos anteriormente. Coloquei como "categoria_desconhecida", já que não temos como, viavelmente, corrigir manualmente.

In [24]:
products["product_category_name"] = products["product_category_name"].fillna("categoria_desconhecida")

Tratar os 2 produtos que encontramos sem peso/dimensões, já que não faz sentido mantê-los pela pouca quantidade de informação real perdida.

In [25]:
products = products[products["product_weight_g"].notna()]

Remover as reviews duplicadas, no caso, mantendo a mais recente ordenando-á

In [26]:
orderReviews = orderReviews.sort_values("review_creation_date").drop_duplicates("order_id", keep="last")

Converter a string para data já pensando na possível feature, pensando em uma taxa de atraso do vendedor.

In [27]:
entregues["order_purchase_timestamp"] = pd.to_datetime(entregues["order_purchase_timestamp"])

Agora, um dos tópicos mais importantes aqui e que nos baseamos muito com a documentação (docs/olist_modelo_relacionamento), em realizando os cruzamentos para a unificação em um dataset final.

In [28]:
df_final = pd.merge(entregues, orderItems, on='order_id', how='left')
df_final = pd.merge(df_final, products, on='product_id', how='left')
df_final = pd.merge(df_final, nameTranslation, on='product_category_name', how='left')
df_final = pd.merge(df_final, customers, on='customer_id', how='left')
df_final = pd.merge(df_final, sellers, on='seller_id', how='left')
df_final = pd.merge(df_final, orderReviews[["order_id","review_score"]], on='order_id', how='left')

pay_principal = orderPayments.sort_values("payment_sequential").groupby("order_id").first().reset_index()
df_final = pd.merge(df_final, pay_principal[["order_id","payment_type","payment_installments"]], on='order_id', how='left')

Apenas uma necessidade, aparentemente o código de baixo para a exportação em .parquet depende dessa linha de código. É uma dor de cabeça que preferi evitar.

In [29]:
!pip install fastparquet

Agora, vamos exportar a base de dados após o pré-processamento e unificada.

In [30]:
df_final.to_csv("../data/processed/base_unificada.csv", index=False)